In [1]:
from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_pymupdf4llm import PyMuPDF4LLMParser

loader = GenericLoader(
    blob_loader=FileSystemBlobLoader(
        path="documents/",
        glob="*.pdf",
    ),
    blob_parser=PyMuPDF4LLMParser(),
)

In [2]:
docs = loader.load()

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


In [3]:
len(docs)

273

In [1]:
import getpass
import os

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API key: ")

In [26]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


In [27]:
from langchain_chroma import Chroma

In [7]:
from tqdm import tqdm

In [2]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model

In [8]:
# Vector store
vector_store = Chroma(
    collection_name="whitepapers",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

# --- Batch insert: 50 docs per batch ---
batch_size = 10
total_docs = len(docs)

# tqdm progress bar with total batches
num_batches = (total_docs + batch_size - 1) // batch_size

for batch_index in tqdm(range(num_batches), desc="Indexing documents"):
    start = batch_index * batch_size
    end = start + batch_size

    batch_docs = docs[start:end]
    batch_ids = [f"doc_{i}" for i in range(start, start + len(batch_docs))]

    vector_store.add_documents(documents=batch_docs, ids=batch_ids)

Indexing documents: 100%|██████████| 28/28 [00:08<00:00,  3.31it/s]


In [11]:
import os
import getpass
from typing import List

from tqdm import tqdm

from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_pymupdf4llm import PyMuPDF4LLMParser

from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document


ROOT_DIR = "documents/"

# 1) Discover all PDF files
def discover_pdfs(root: str) -> List[str]:
    pdfs = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dirpath, f))
    return sorted(pdfs)


# 2) Set up API key + embeddings + Chroma
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_store = Chroma(
    collection_name="whitepapers_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)


# 3) File-level batching: N files at a time
file_batch_size = 2         # 📂 how many PDFs per batch
chunk_batch_size = 100        # 📄 how many chunks per add_documents call

all_pdf_paths = discover_pdfs(ROOT_DIR)
total_files = len(all_pdf_paths)
print(f"Found {total_files} PDF files.")

for file_start in tqdm(
    range(0, total_files, file_batch_size),
    desc="Processing file batches",
    unit="batch",
):
    file_end = min(file_start + file_batch_size, total_files)
    batch_paths = all_pdf_paths[file_start:file_end]

    # ---- Load + chunk these N files into Documents ----
    batch_docs: List[Document] = []
    for path in batch_paths:
        loader = GenericLoader(
            blob_loader=FileSystemBlobLoader(
                path=os.path.dirname(path),
                glob=os.path.basename(path),
            ),
            blob_parser=PyMuPDF4LLMParser(),
        )
        docs = loader.load()   # or `await loader.aload()` if you're in async context

        # Enrich metadata with file-level info
        for i, d in enumerate(docs):
            d.metadata = {
                **(d.metadata or {}),
                "source": path,
                "chunk_index": i,
            }

        batch_docs.extend(docs)

    if not batch_docs:
        continue

    # ---- Now ingest these chunks in sub-batches ----
    for start in tqdm(
        range(0, len(batch_docs), chunk_batch_size),
        desc=f"Indexing chunks for files {file_start+1}-{file_end}",
        leave=False,
        unit="chunk",
    ):
        end = start + chunk_batch_size
        sub_docs = batch_docs[start:end]

        sub_ids = [
            f"{os.path.basename(d.metadata.get('source', 'unknown'))}::chunk_{d.metadata.get('chunk_index', i)}"
            for i, d in enumerate(sub_docs)
        ]

        vector_store.add_documents(documents=sub_docs, ids=sub_ids)

# Optionally force persistence
# vector_store.persist()
print("Ingestion complete.")

Found 5 PDF files.


Processing file batches: 100%|██████████| 3/3 [02:51<00:00, 57.13s/batch]

Ingestion complete.


In [ ]:
import os
import getpass
import hashlib
from typing import List

from tqdm import tqdm

from langchain_community.document_loaders import FileSystemBlobLoader
from langchain_community.document_loaders.generic import GenericLoader
from langchain_pymupdf4llm import PyMuPDF4LLMParser
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


# -----------------------
# CONFIG
# -----------------------

ROOT_DIR = "documents/"
PERSIST_DIR = "./chroma_langchain_db"
COLLECTION_NAME = "whitepapers_new"

FILE_BATCH_SIZE = 50   # how many PDFs we process at a time
CHUNK_BATCH_SIZE = 200 # how many chunks per add_documents() call


# -----------------------
# HELPERS
# -----------------------

def discover_pdfs(root: str) -> List[str]:
    pdfs: List[str] = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".pdf"):
                pdfs.append(os.path.join(dirpath, f))
    return sorted(pdfs)


def compute_file_id(path: str) -> str:
    """Stable ID from file path (used in vector IDs)."""
    return hashlib.md5(path.encode("utf-8")).hexdigest()


def split_modalities(doc: Document) -> List[Document]:
    """
    Very simple heuristic splitter:
    - markdown tables -> modality='table'
    - image-like markdown / tags -> modality='image'
    - everything else -> modality='text'
    All stay as text, just tagged differently in metadata.
    """
    text_blocks = []
    table_blocks = []
    image_blocks = []

    blocks = doc.page_content.split("\n\n")
    for block in blocks:
        b = block.strip()
        if not b:
            continue
        if b.startswith("|") and "---" in b:
            table_blocks.append(block)
        elif b.startswith("![") or "<img" in b:
            image_blocks.append(block)
        else:
            text_blocks.append(block)

    out_docs: List[Document] = []

    if text_blocks:
        out_docs.append(
            Document(
                page_content="\n\n".join(text_blocks),
                metadata={**doc.metadata, "modality": "text"},
            )
        )

    for tb in table_blocks:
        out_docs.append(
            Document(
                page_content=tb,
                metadata={**doc.metadata, "modality": "table"},
            )
        )

    for ib in image_blocks:
        out_docs.append(
            Document(
                page_content=ib,
                metadata={**doc.metadata, "modality": "image"},
            )
        )

    # If nothing matched the heuristics, at least return the original as mixed
    if not out_docs:
        out_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={**doc.metadata, "modality": "mixed"},
            )
        )

    return out_docs


# -----------------------
# MAIN INGESTION
# -----------------------

def main():
    # 1) Discover files
    pdf_paths = discover_pdfs(ROOT_DIR)
    total_files = len(pdf_paths)
    print(f"Found {total_files} PDF files under {ROOT_DIR}")

    if total_files == 0:
        return

    # 2) API key
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

    # 3) Embeddings + Chroma
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=PERSIST_DIR,
    )

    # 4) Multimodal parser: tables + images
    images_parser = RapidOCRBlobParser()
    parser = PyMuPDF4LLMParser(
        extract_images=True,
        images_parser=images_parser,
        table_strategy="lines",   # you can adjust to "lines_strict" / "text" etc.
    )

    # 5) Process files in batches
    for file_start in tqdm(
        range(0, total_files, FILE_BATCH_SIZE),
        desc="Processing file batches",
        unit="batch",
    ):
        file_end = min(file_start + FILE_BATCH_SIZE, total_files)
        batch_paths = pdf_paths[file_start:file_end]

        batch_docs: List[Document] = []

        # ---- load + chunk these files ----
        for path in batch_paths:
            file_id = compute_file_id(path)

            loader = GenericLoader(
                blob_loader=FileSystemBlobLoader(
                    path=os.path.dirname(path),
                    glob=os.path.basename(path),
                ),
                blob_parser=parser,
            )
            try:
                docs = loader.load()
            except Exception as e:
                print(f"\nError loading {path}: {e}")
                continue

            # enrich & split modalities
            for i, d in enumerate(docs):
                base_meta = {
                    **(d.metadata or {}),
                    "source": path,
                    "file_id": file_id,
                    "chunk_index": i,
                }
                d.metadata = base_meta
                # split into text/table/image flavored docs
                modal_docs = split_modalities(d)
                batch_docs.extend(modal_docs)

        if not batch_docs:
            continue

        # ---- ingest chunks in sub-batches ----
        for start in tqdm(
            range(0, len(batch_docs), CHUNK_BATCH_SIZE),
            desc=f"Indexing chunks for files {file_start+1}-{file_end}",
            leave=False,
            unit="chunk",
        ):
            end = start + CHUNK_BATCH_SIZE
            sub_docs = batch_docs[start:end]

            sub_ids = [
                f"{d.metadata.get('file_id', 'unknown')}::chunk_{d.metadata.get('chunk_index', i)}::mod_{d.metadata.get('modality', 'mixed')}"
                for i, d in enumerate(sub_docs)
            ]

            vector_store.add_documents(documents=sub_docs, ids=sub_ids)

    # 6) Persist
    # vector_store.persist()
    print("Ingestion complete.")


if __name__ == "__main__":
    main()

Found 5 PDF files under documents/


Processing file batches:   0%|          | 0/1 [00:00<?, ?batch/s]Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmpqaae4iz4/Agent-Tools-&-Interoperability-with-Model-Context-Protocol-(MCP
Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmp5j9ck0m8/Agent-Tools-&-Interoperability-with-Model-Context-Protocol-(MCP
Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmp57gctpoh/Agent-Tools-&-Interoperability-with-Model-Context-Protocol-(MCP
Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmplpvrgyq2/Agent-Tools-&-Interoperability-with-Model-Context-Protocol-(MCP
Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmp1hcfbu73/Agent-Tools-&-Interoperability-with-Model-Context-Protocol-(MCP
Image path referenced in markdown but not found: C:/Users/Sangavi/AppData/Local/Temp/tmpg180cuzg/Agent-Tools-&-Interoperability-wi

Ingestion complete.


In [15]:
import os
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# 2) Set up API key + embeddings + Chroma
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Vector store
vector_store = Chroma(
    collection_name="whitepapers",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",
)

similar_docs = vector_store.similarity_search("What is observability?", k=2)
similar_docs

[Document(id='doc_31', metadata={'total_pages': 51, 'modDate': "D:20251110084739-07'00'", 'page': 31, 'author': '', 'keywords': '', 'title': '', 'creator': 'Adobe InDesign 20.5 (Macintosh)', 'subject': '', 'creationdate': '2025-11-10T08:47:32-07:00', 'producer': 'Adobe PDF Library 17.0', 'source': 'documents\\Agent Quality.pdf', 'file_path': 'documents\\Agent Quality.pdf', 'format': 'PDF 1.7', 'moddate': '2025-11-10T08:47:39-07:00', 'creationDate': "D:20251110084732-07'00'", 'trapped': ''}, page_content='Agent Quality\n\n\n**An AI Agent is a Gourmet Chef in a "Mystery Box" Challenge:** The chef is given a\n\ngoal ("Create an amazing dessert") and a basket of ingredients (the user\'s prompt, data,\n\nand available tools). There is no single correct recipe. They might create a chocolate lava\n\ncake, a deconstructed tiramisu, or a saffron-infused panna cotta. All could be valid, even\n\n\nbrilliant, solutions.\n\n\n**• Observability** is how a food critic would judge the chef. The critic

In [5]:
@tool
def rag_context_retriever(query: str, k: int = 4):
    """Retrieve top-k relevant documents for RAG context.
    Args:
        query (str): The input query string.
        k (int): Number of similar documents to retrieve.
    Returns:
        Formatted Context.
    """
    similar_docs = vector_store.similarity_search(query, k=k)
    contexts = [doc.page_content for doc in similar_docs]
    formatted_contexts = "\n\n---\n\n".join(contexts)
    return formatted_contexts

In [4]:
# print(rag_context_retriever("Explain observability in IT systems.", k=3))

In [9]:


model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    temperature=0
)

In [10]:
model.invoke("hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b0017-9b45-7092-b03c-c7a55b47612c-0', usage_metadata={'input_tokens': 2, 'output_tokens': 195, 'total_tokens': 197, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 185}})

In [11]:
tools = [rag_context_retriever]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = model.bind_tools(tools)

In [12]:
llm_with_tools.invoke("Explain observability in IT systems.")

AIMessage(content=[{'type': 'text', 'text': "Observability in IT systems refers to the ability to understand the internal state of a system by examining its external outputs. It's about being able to ask arbitrary questions about your system without having to deploy new code to answer those questions.\n\nThis is typically achieved through the collection and analysis of three main types of telemetry data:\n\n1.  **Logs:** These are timestamped records of discrete events that happen within a system. They provide detailed context about what occurred at a specific point in time, such as errors, warnings, or informational messages.\n2.  **Metrics:** These are numerical measurements collected over time, representing the health and performance of a system or its components. Examples include CPU usage, memory consumption, request rates, and error rates. Metrics are useful for identifying trends and anomalies.\n3.  **Traces:** These represent the end-to-end journey of a request or transaction a

In [13]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

class QueryAnalysis(BaseModel):
    """Structured output for the Query Analyzer node."""
    # Optional rewritten / normalized query
    rewritten_query: Optional[str] = Field(
        default=None,
        description="A clarified, normalized version of the user query. Only provided when safe to rewrite.",
    )

    # Clarification decision
    needs_clarification: bool = Field(
        ...,
        description="Whether the system must ask a clarification question before proceeding.",
    )

    # Optional clarification questions
    clarification_questions: Optional[List[str]] = Field(
        default=None,
        description="Clarification questions to ask the user if needs_clarification is True.",
    )

    # Optional reasoning fields
    rationale: Optional[str] = Field(
        default=None,
        description="Explanation for reasoning, rewrite choice, and clarification decision.",
    )

    intent: Literal[
        "information_retrieval",
        "greetings_or_generic",
    ] = Field(
        default="greetings_or_generic",
        description="High-level intent classification of the query.",
    )

    confidence: float = Field(
        default=0.0,
        ge=0.0,
        le=1.0,
        description="Model's confidence (0–1) in its analysis.",
    )

model_with_structured_output = llm_with_tools.with_structured_output(QueryAnalysis)
model_with_structured_output.invoke("Analyze the following user query and provide structured output according to the QueryAnalysis schema.\nUSER QUERY: What is observability")

QueryAnalysis(rewritten_query=None, needs_clarification=False, clarification_questions=None, rationale='The user is asking for a definition, which is a direct information retrieval task. The query is clear and does not require clarification or rewriting.', intent='information_retrieval', confidence=0.95)

In [ ]:
QUERY_ANALYZER_PROMPT = """You are a Query Analyzer for a retrieval-based assistant.

Your job is to:
1) Decide whether the user’s query needs clarification before proceeding.
2) Optionally rewrite the query into a clearer, retrieval-friendly form.
3) Classify the intent as either:
   - "information_retrieval"
   - "greetings_or_generic"
4) Provide a confidence score between 0 and 1.

You MUST follow these rules when filling the QueryAnalysis schema:

- rewritten_query:
  - Use this to produce a concise, unambiguous version of the user query that is suitable for retrieval (search / RAG).
  - Keep the original meaning; do NOT add constraints or assumptions that the user did not state.
  - If the query is a pure greeting or generic chit-chat, set rewritten_query to null.
  - If the query is too ambiguous and truly cannot be safely rewritten yet, set rewritten_query to null.

- needs_clarification:
  - Set to True only if the system cannot safely act on the query without more information, and a follow-up question is required.
  - Common reasons:
    - Missing key constraints (e.g., timeframe, dataset, document, product, location) that are essential for a meaningful answer.
    - Multiple distinct interpretations or tasks that must be disambiguated.
  - If you set needs_clarification = True:
    - clarification_questions MUST contain one or more short, direct questions.
    - Each question should ask for only one thing.
    - Do NOT rewrite the query; leave rewritten_query as null or a very generic version if needed.

- clarification_questions:
  - Only populate if needs_clarification = True.
  - Ask the minimum number of questions needed to make the query actionable (usually 1–3).
  - Questions must be natural and user-facing, e.g. "Which time period are you interested in?" 

- intent:
  - "information_retrieval":
      The user is asking for information, explanation, comparison, summary, or similar content that can be answered via retrieval or knowledge.
      Examples: "What is observability?", "Summarize the latest 5G whitepaper", "Compare Kafka and Pub/Sub."
  - "greetings_or_generic":
      Greetings, small talk, thanks, or meta-questions about the assistant.
      Examples: "Hi", "Hello", "How are you?", "Thanks!", "Who are you?"

- confidence:
  - 1.0 → very clear, straightforward query and classification.
  - ~0.5 → somewhat ambiguous, but you can still make a reasonable judgment.
  - <0.3 → highly ambiguous or unclear query.

- rationale:
  - Optional short explanation (1–3 sentences) of why you decided on needs_clarification, intent, and (if present) the rewritten_query.

General behavior:
- Be conservative about rewriting: avoid adding details the user did not provide.
- Prefer needs_clarification = False if the query is reasonably understandable and can be answered broadly.
- Do NOT ask clarification if you can reasonably answer the query as-is with a generic interpretation.

Now analyze the following user query and produce a QueryAnalysis object:

CHAT_HISTORY:
{chat_history}

USER QUERY:
{user_query}
"""

In [ ]:
analysis = model_with_structured_output.invoke(
    QUERY_ANALYZER_PROMPT.format(chat_history=state[],user_query="what is?")
)

In [ ]:
analysis.model_dump()

{'rewritten_query': None,
 'needs_clarification': True,
 'clarification_questions': ['What topic are you interested in?',
  'What information are you looking for?'],
 'rationale': "The query 'what is?' is too vague and incomplete to understand the user's intent without further clarification. It's an information retrieval intent, but the subject is missing.",
 'intent': 'information_retrieval',
 'confidence': 0.2}

: 

In [ ]:
from typing import TypedDict, Optional, List
from typing_extensions import NotRequired
from langchain_core.documents import Document
from langchain_core.messages import AnyMessage
from langgraph.graph import MessagesState, StateGraph, START, END
import operator
from typing import Annotated

class AgentState(MessagesState):
    # Core query info
    user_query: str                         # original user input
    rewritten_query: NotRequired[str]       # from QueryAnalyzer, if available

    # # Conversation / agent messages (optional but recommended for agents)
    # messages: NotRequired[
    #     Annotated[List[AnyMessage], operator.add]
    # ]

    # Retrieval-related
    retrieved_context: NotRequired[List[Document]]  # raw retrieved docs or chunks

    # Clarification control
    clarification_counter: int              # to cap number of clarification turns

    # Final answer from the agent
    final_response: NotRequired[str]

    # Evaluation / QA
    response_quality_score: NotRequired[float]
    response_quality_score_reason: NotRequired[str]